# 10 — IoU Threshold Sensitivity: τ=0.50 vs τ=0.25 (DIAGNOSTIC)

> **Temporary diagnostic tool — not part of the final pipeline.**  
> **Scope: SpaceNet cities only** (raw building files available on Drive).

Compares city-level F1 at:

| Threshold | Source |
|-----------|--------|
| **τ = 0.50** | Stored tile metrics (pipeline default, PASCAL-VOC standard) |
| **τ = 0.25** | Re-run tile-level matching on raw building files (SpaceNet benchmark standard) |

All other parameters are kept identical (buffer = `tau_buffer_m` from config, min area, CRS).

**Why re-run?** The pipeline match parquets only store pairs with IoU ≥ 0.50.  
Computing F1 at τ=0.25 requires finding the extra pairs (0.25 ≤ IoU < 0.50) that the  
pipeline discarded, so matching must be re-run from the raw geometries.

**Why SpaceNet only?** The τ=0.25 threshold originates from the SpaceNet benchmark.  
The comparison is most meaningful for cities where the reference data is SpaceNet7.  
Non-SpaceNet cities are also the ones most likely to be missing raw building files.

**Output:** `outputs/scratch/iou_threshold_sensitivity.csv`

In [ ]:
!pip install -q geopandas shapely
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 1 — Setup & load τ=0.50 baseline from stored tile metrics ────────────
import sys, time
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml

CONFIG_PATH  = Path('/content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/configs/validation_configs.yaml')
PROJECT_ROOT = CONFIG_PATH.parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
cfg['root_dir'] = str(PROJECT_ROOT)

DATA_DIR     = PROJECT_ROOT / cfg.get('data_dir', 'data/01_raw')
METRICS_ROOT = PROJECT_ROOT / 'outputs' / 'metrics'
SCRATCH_DIR  = PROJECT_ROOT / 'outputs' / 'scratch'
SENS_DIR     = PROJECT_ROOT / 'outputs' / 'sensitivity_studies'
SENS_DIR.mkdir(parents=True, exist_ok=True)
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)

vec_pre        = cfg['vector']['preprocessing']
TAU_PIPELINE   = float(vec_pre.get('iou_threshold', vec_pre.get('tau_overlap', 0.5)))
TAU_BUFFER_M   = float(vec_pre.get('tau_buffer_m', 2.0))
MIN_AREA_M2    = float(vec_pre.get('min_area_m2', 20.0))
FIX_GEOMS      = bool(vec_pre.get('fix_invalid_geoms', True))
TAU_COMPARE    = 0.25   # SpaceNet benchmark standard

TILE_SENTINEL  = 'vector_metrics_tiles_all_datasets.parquet'

print(f'Pipeline IoU threshold : {TAU_PIPELINE}')
print(f'Comparison threshold   : {TAU_COMPARE}')
print(f'Buffer                 : {TAU_BUFFER_M} m  (kept identical for both runs)')

# ── SpaceNet7 flag ────────────────────────────────────────────────────────────
TRACKER_PATH = PROJECT_ROOT / 'data/02_interim/aoi_tracker.csv'
ref_source_map = {}
ref_file_map   = {}

if TRACKER_PATH.exists():
    tracker = pd.read_csv(TRACKER_PATH, dtype=str)
    tracker.columns = tracker.columns.str.strip()
    id_col = 'dataset_folder_name'
    if 'reference_source' in tracker.columns:
        ref_source_map = (
            tracker.dropna(subset=[id_col])
            .set_index(id_col)['reference_source']
            .str.strip().str.lower().to_dict()
        )
    ref_col = next(
        (c for c in tracker.columns if 'reference' in c.lower() and 'file' in c.lower()), None
    )
    if ref_col:
        for _, row in tracker.dropna(subset=[id_col]).iterrows():
            city = str(row[id_col]).strip()
            raw  = str(row.get(ref_col, '') or '')
            parts = [p.strip() for p in raw.split('|') if p.strip()]
            if parts:
                ref_file_map.setdefault(city, []).extend(parts)
    sn7 = sum(1 for v in ref_source_map.values() if v == 'spacenet')
    print(f'Tracker loaded: {len(ref_source_map)} cities  ({sn7} SpaceNet7)')
else:
    print(f'[WARN] Tracker not found at {TRACKER_PATH}')

# ── Restrict to SpaceNet cities ───────────────────────────────────────────────
# τ=0.25 is the SpaceNet benchmark standard; the comparison is most meaningful
# for cities with SpaceNet7 reference data. Non-SpaceNet cities are also the
# ones most likely to be missing raw building files on Drive.
CITY_SUBSET = sorted(c for c, src in ref_source_map.items() if src == 'spacenet')
if CITY_SUBSET:
    print(f'\nRunning for {len(CITY_SUBSET)} SpaceNet cities: {CITY_SUBSET}')
else:
    print('\n[WARN] No SpaceNet cities found in tracker — falling back to all cities.')
    CITY_SUBSET = None

# ── Load τ=0.50 baseline from stored tile metrics ─────────────────────────────
baseline_rows = []
city_dirs_all = sorted(p.parent for p in METRICS_ROOT.rglob(TILE_SENTINEL))
for city_dir in city_dirs_all:
    city = city_dir.name
    if CITY_SUBSET and city not in CITY_SUBSET:
        continue
    try:
        tile_df = pd.read_parquet(city_dir / TILE_SENTINEL)
        for ds, g in tile_df.groupby('dataset'):
            tp = int(g['tp'].sum()); fp = int(g['fp'].sum()); fn = int(g['fn'].sum())
            p  = tp / (tp + fp) if (tp + fp) else 0.0
            r  = tp / (tp + fn) if (tp + fn) else 0.0
            f1 = 2*p*r / (p+r) if (p+r) else 0.0
            baseline_rows.append({
                'city': city, 'dataset': ds,
                'is_spacenet7': True,
                f'f1_iou{int(TAU_PIPELINE*100):02d}': round(f1, 4),
            })
    except Exception as e:
        print(f'  [WARN] {city}: {e}')

df_baseline = pd.DataFrame(baseline_rows)
print(f'\nBaseline (τ={TAU_PIPELINE}): {len(df_baseline)} rows | {df_baseline["city"].nunique()} cities')
print('Cell 1 done.')

In [ ]:
# ── Cell 2 — Re-run matching at τ=0.25 using raw building files ───────────────
#
# Loads raw reference + candidate buildings and tiles for each city,
# runs match_buildings_iou() at TAU_COMPARE (0.25) with the same buffer.
# Cities where raw data is unavailable are skipped.

from src.metrics.vector.matching import match_buildings_iou
from src.utils.buildings import load_buildings
from src.utils.tiling import subset_by_tile

skipped = []
recomp_rows = []

cities_to_run = df_baseline['city'].unique().tolist()
print(f'Re-running matching at τ={TAU_COMPARE} for {len(cities_to_run)} cities...')
print(f'(buffer kept at {TAU_BUFFER_M} m)\n')


def find_reference_files(city):
    if city in ref_file_map:
        paths = [DATA_DIR / city / 'vector' / f for f in ref_file_map[city]]
        return [p for p in paths if p.exists()]
    # fallback: any gpkg in vector/ that isn't a known candidate
    vec_dir = DATA_DIR / city / 'vector'
    if not vec_dir.exists():
        return []
    cand_names = {'overture', 'gba', 'globfp'}
    return [f for f in vec_dir.glob('*.gpkg')
            if not any(n in f.stem.lower() for n in cand_names)]


def find_candidate_files(city, ds_name):
    city_slug = city.lower()
    vec_dir   = DATA_DIR / city / 'vector'
    pattern   = f'{city_slug.replace("-", "_")}_{ds_name}*.parquet'
    return sorted(vec_dir.glob(pattern))


def city_f1_at_tau(ref_all, cand_all, tiles, tau):
    ref_idx  = ref_all.sindex
    cand_idx = cand_all.sindex
    tp = fp = fn = 0
    for tile_row in tiles.itertuples():
        ref_tile  = subset_by_tile(ref_all,  ref_idx,  tile_row.geometry)
        cand_tile = subset_by_tile(cand_all, cand_idx, tile_row.geometry)
        if ref_tile.empty and cand_tile.empty:
            continue
        matches, ref_unm, cand_unm = match_buildings_iou(
            ref_tile, cand_tile, tau, tau_buffer_m=TAU_BUFFER_M
        )
        tp += len(matches); fp += len(cand_unm); fn += len(ref_unm)
    p  = tp / (tp + fp) if (tp + fp) else 0.0
    r  = tp / (tp + fn) if (tp + fn) else 0.0
    return round(2*p*r / (p+r) if (p+r) else 0.0, 4)


for city in cities_to_run:
    t0 = time.time()
    ref_paths = find_reference_files(city)
    if not ref_paths:
        skipped.append((city, 'reference files not found'))
        continue

    city_slug  = city.lower()
    tiles_path = DATA_DIR / city / 'tiles' / f'{city_slug}_tiles.gpkg'
    if not tiles_path.exists():
        skipped.append((city, 'tiles GPKG not found'))
        continue

    try:
        tiles = gpd.read_file(tiles_path)
        crs   = tiles.crs.to_string()
    except Exception as e:
        skipped.append((city, f'tiles load error: {e}'))
        continue

    try:
        ref_parts = [load_buildings(p, crs_work=crs, min_area_m2=MIN_AREA_M2,
                                    fix_invalid_geoms=FIX_GEOMS) for p in ref_paths]
        ref_all = (gpd.GeoDataFrame(pd.concat(ref_parts, ignore_index=True),
                                    crs=ref_parts[0].crs)
                   if len(ref_parts) > 1 else ref_parts[0])
    except Exception as e:
        skipped.append((city, f'reference load error: {e}'))
        continue

    ds_names = df_baseline[df_baseline['city'] == city]['dataset'].unique()
    any_ran  = False
    for ds_name in ds_names:
        cand_files = find_candidate_files(city, ds_name)
        if not cand_files:
            continue
        try:
            cand_all = load_buildings(cand_files[0], crs_work=crs,
                                      min_area_m2=MIN_AREA_M2, fix_invalid_geoms=FIX_GEOMS)
        except Exception as e:
            print(f'  [WARN] {city}/{ds_name}: {e}')
            continue

        f1_025 = city_f1_at_tau(ref_all, cand_all, tiles, TAU_COMPARE)
        recomp_rows.append({'city': city, 'dataset': ds_name, 'f1_iou25': f1_025})
        any_ran = True

    elapsed = time.time() - t0
    print(f'  {city:<40} {"ok" if any_ran else "no candidates"}  ({elapsed:.0f}s)')

df_recomp = pd.DataFrame(recomp_rows)

if skipped:
    print(f'\nSkipped {len(skipped)} cities (raw data not on Drive):')
    for city, reason in skipped[:5]:
        print(f'  {city}: {reason}')
    if len(skipped) > 5:
        print(f'  ... and {len(skipped)-5} more')

print(f'\nRecomputed τ=0.25: {len(df_recomp)} rows')

In [ ]:
# ── Cell 3 — Comparison table + box plots ────────────────────────────────────

# Merge baseline (τ=0.50) with recomputed (τ=0.25)
df = df_baseline.merge(df_recomp, on=['city', 'dataset'], how='left')
df['delta_025_vs_50'] = (df['f1_iou25'] - df['f1_iou50']).round(4)

GROUP_PALETTE = {'SpaceNet7': '#0072B2', 'Non-SpaceNet': '#E69F00'}
df['group'] = df['is_spacenet7'].map({True: 'SpaceNet7', False: 'Non-SpaceNet'})

# ── Summary table ─────────────────────────────────────────────────────────────
print('=== F1 at τ=0.50 vs τ=0.25 (mean across datasets per city) ===')
df_city = (
    df.groupby(['city', 'is_spacenet7', 'group'])[
        ['f1_iou50', 'f1_iou25', 'delta_025_vs_50']
    ].mean().reset_index().round(4)
)
print(f'Cities with both values: {df_city["f1_iou25"].notna().sum()} / {len(df_city)}\n')
for grp_label, grp in df_city.groupby('group'):
    grp_valid = grp.dropna(subset=['f1_iou25'])
    print(f'  {grp_label} ({len(grp_valid)} cities):')
    for col in ['f1_iou50', 'f1_iou25', 'delta_025_vs_50']:
        if grp_valid[col].notna().any():
            print(f'    {col:<22}  mean={grp_valid[col].mean():.4f}  '
                  f'median={grp_valid[col].median():.4f}  std={grp_valid[col].std():.4f}')
    print()

# ── Plots ─────────────────────────────────────────────────────────────────────
df_plot = df_city.dropna(subset=['f1_iou25'])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 1: F1 at τ=0.50 vs τ=0.25 side by side
df_melt = df_plot.melt(
    id_vars=['city', 'group'],
    value_vars=['f1_iou50', 'f1_iou25'],
    var_name='threshold', value_name='f1'
)
df_melt['threshold_label'] = df_melt['threshold'].map(
    {'f1_iou50': 'τ = 0.50\n(pipeline)', 'f1_iou25': 'τ = 0.25\n(SpaceNet)'})

sns.boxplot(data=df_melt, x='threshold_label', y='f1', hue='group',
            palette=GROUP_PALETTE, ax=axes[0], linewidth=1.2)
axes[0].set_title('F1: pipeline (τ=0.50) vs SpaceNet (τ=0.25)', fontweight='bold')
axes[0].set_xlabel('IoU threshold')
axes[0].set_ylabel('City-level F1')
axes[0].set_ylim(0, 1.05)
axes[0].legend(title='Group', fontsize=8)

# Panel 2: Δ F1 (τ=0.25 minus τ=0.50) — how much the more lenient threshold gains
sns.boxplot(data=df_plot, x='group', y='delta_025_vs_50',
            palette=GROUP_PALETTE, ax=axes[1], linewidth=1.2)
axes[1].axhline(0, color='grey', linestyle='--', linewidth=1)
axes[1].set_title('Δ F1: τ=0.25 minus τ=0.50\n(positive = lenient threshold gains F1)',
                  fontweight='bold')
axes[1].set_xlabel('Group')
axes[1].set_ylabel('ΔF1 (τ0.25 − τ0.50)')

# Panel 3: Scatter τ=0.50 vs τ=0.25 — one dot per city × dataset
DATASET_MARKERS = {'overture': 'o', 'gba': 's', 'globfp': '^'}
df_scatter = df.dropna(subset=['f1_iou25'])
for ds, ds_grp in df_scatter.groupby('dataset'):
    for grp_label, g in ds_grp.groupby('group'):
        axes[2].scatter(
            g['f1_iou50'], g['f1_iou25'],
            color=GROUP_PALETTE[grp_label],
            marker=DATASET_MARKERS.get(ds, 'o'),
            alpha=0.4, s=14,
            label=f'{ds}' if grp_label == 'SpaceNet7' else '_nolegend_'
        )
axes[2].plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='y = x (no change)')
axes[2].set_xlabel('F1 at τ = 0.50')
axes[2].set_ylabel('F1 at τ = 0.25')
axes[2].set_title('Scatter per city × dataset\n(above y=x = τ=0.25 gives higher F1)',
                  fontweight='bold')
axes[2].legend(fontsize=8, title='Dataset')

for ax in axes:
    ax.grid(axis='y', alpha=0.3)
    sns.despine(ax=ax)

fig.suptitle(
    f'IoU threshold sensitivity — pipeline τ=0.50 vs SpaceNet τ=0.25  '
    f'(buffer = {TAU_BUFFER_M} m, unchanged)',
    fontsize=13, fontweight='bold', y=1.01
)
fig.tight_layout()
out_fig = SCRATCH_DIR / 'iou_threshold_sensitivity_boxplot.png'
fig.savefig(out_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved → {out_fig}')

In [ ]:
# ── Cell 4 — Save CSV ─────────────────────────────────────────────────────────
out_path = SENS_DIR / 'iou_threshold_sensitivity.csv'
df.to_csv(out_path, index=False)

print(f'Saved → {out_path}')
print(f'  {len(df):,} rows  |  columns: {list(df.columns)}')
print(f'  Cities with τ=0.25 data : {df["f1_iou25"].notna().sum() // df["dataset"].nunique()}')
print(f'  Cities skipped (no data): {len(skipped)}')
print()
print('=== Global means by dataset ===')
display(
    df.dropna(subset=['f1_iou25'])
    .groupby('dataset')[['f1_iou50', 'f1_iou25', 'delta_025_vs_50']]
    .mean().round(4)
)